In [4]:
import pandas as pd
import numpy as np

FILE = "data/raw/expense_claims.csv"
df = pd.read_csv(FILE, dtype=str, keep_default_na=False,na_values=["", "NULL", "null"])
print("rows, cols:", df.shape)
df.head(3)

rows, cols: (83, 80)


,item_pk,item_id,report_pk,employee_id,report_id,employee_id-2,expense_type_name,expense_category_name,expense_category_code,bill_number,...,latest_step_caller_auth_user_id,latest_step_remark_text,is_duplicate_flagged,duplicate_match_count,override_count,last_override_reason,last_overridden_at,item_created_on,item_updated_on,submission_date
0,888,e-31,361,794,EXP-361,794,D2,DUP,e,NaN,...,NaN,NaN,FALSE,0,0,NaN,NaN,2026-07-16 10:19:12,2026-07-16 10:23:47,2026-07-16 10:19:44
1,885,LEARN-10,359,794,EXP-359,794,Certificates,Learning,LEARN,NaN,...,NaN,NaN,FALSE,0,0,NaN,NaN,2026-07-16 10:11:57,2026-07-16 10:12:23,2026-07-16 10:12:22
2,884,LEARN-9,359,794,EXP-359,794,Certificates,Learning,LEARN,NaN,...,NaN,NaN,FALSE,0,0,NaN,NaN,2026-07-16 10:11:05,2026-07-16 10:12:23,2026-07-16 10:12:22


In [5]:
for i, c in enumerate(df.columns):
    print(f"{i}  {c}")

0  item_pk
1  item_id
2  report_pk
3  employee_id
4  report_id
5  employee_id-2
6  expense_type_name
7  expense_category_name
8  expense_category_code
9  bill_number
10  bill_date
11  bill_amount
12  tax_amount
13  amount_before_tax
14  bill_currency_id
15  payout_currency_id
16  exchange_rate
17  payout_amount
18  overridden_amount
19  overridden_tax_amount
20  overridden_reason
21  has_attachment
22  mileage_unit
23  mileage_quantity
24  mileage_rate_per_unit
25  mileage_original_rate_per_unit
26  mileage_overridden_quantity
27  item_status
28  report_status
29  auto_approved
30  approval_date
31  rejection_date
32  policy_id
33  rule_id
34  limit_per_claim
35  limit_per_day
36  limit_per_month
37  limit_per_quarter
38  limit_per_year
39  allow_beyond_limit
40  limit_scope
41  frequency_per_day
42  frequency_per_month
43  frequency_per_quarter
44  frequency_per_year
45  allow_beyond_frequency
46  configured_reviewer_chain
47  configured_reviewer_count
48  latest_remark
49  remark_cou

In [6]:
pd.set_option("display.max_rows", 100)
pd.DataFrame({
    "nulls":    df.isna().sum(),
    "null_%":   (df.isna().mean() * 100).round(1),
    "distinct": df.nunique(),
})

,nulls,null_%,distinct
item_pk,0,0.0,83
item_id,0,0.0,83
report_pk,0,0.0,71
employee_id,0,0.0,9
report_id,0,0.0,71
employee_id-2,0,0.0,9
expense_type_name,0,0.0,32
expense_category_name,0,0.0,17
expense_category_code,0,0.0,17
bill_number,32,38.6,40


In [7]:
print(df["item_status"].value_counts(dropna=False))
print()
print("auto_approved:", df["auto_approved"].value_counts(dropna=False).to_dict())
print("override_count > 0:", (df["override_count"].fillna("0") != "0").sum())
print("duplicate flagged:", df["is_duplicate_flagged"].value_counts(dropna=False).to_dict())

item_status
Accepted    63
Rejected    20
Name: count, dtype: int64

auto_approved: {nan: 71, 'TRUE': 12}
override_count > 0: 18
duplicate flagged: {'FALSE': 73, 'TRUE': 10}


In [8]:
pairs = df.groupby(["employee_id", "expense_category_code"]).size() #add expense type
print("total (employee, category) pairs:", len(pairs))
print("claims per pair:")
print(pairs.value_counts().sort_index().head(10))
print("pairs with >= 3 claims:", (pairs >= 3).sum())

total (employee, category) pairs: 32
claims per pair:
1     17
2      7
3      2
5      3
8      2
15     1
Name: count, dtype: int64
pairs with >= 3 claims: 8


In [9]:
assert (df["employee_id"] == df["employee_id-2"]).all(), "employee ids differ"
df = df.drop(columns=["employee_id-2"])
#need not do this
DATE_COLS = [
    "bill_date", "submission_date", "approval_date", "rejection_date",
    "item_created_on", "item_updated_on", "last_overridden_at",
    "last_remark_at", "first_remark_at",
    "latest_step_start_date", "latest_step_due_date", "latest_step_finish_date",
]

NUM_COLS = [
    "bill_amount", "tax_amount", "amount_before_tax", "exchange_rate",
    "payout_amount", "overridden_amount", "overridden_tax_amount",
    "mileage_quantity", "mileage_rate_per_unit",
    "mileage_original_rate_per_unit", "mileage_overridden_quantity",
    "limit_per_claim", "limit_per_day", "limit_per_month",
    "limit_per_quarter", "limit_per_year",
    "frequency_per_day", "frequency_per_month",
    "frequency_per_quarter", "frequency_per_year",
    "duplicate_match_count", "override_count", "remark_count",
    "workflow_step_count", "last_review_level", "configured_reviewer_count",
]

BOOL_COLS = ["has_attachment", "auto_approved", "is_duplicate_flagged",
             "allow_beyond_limit", "allow_beyond_frequency"]

ID_COLS = ["item_pk", "report_pk", "employee_id",
           "bill_currency_id", "payout_currency_id"]

for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], errors="coerce")

for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in BOOL_COLS:
    df[c] = df[c].map({"TRUE": True, "True": True, "true": True,
                       "FALSE": False, "False": False, "false": False})

for c in ID_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")

# stays a string on purpose — int64 would drop leading zeros
df["bill_number"] = df["bill_number"].str.strip()

df["expense_category_code"] = df["expense_category_code"].astype("category")

print(df.dtypes.value_counts())
df[["item_id","employee_id","expense_category_code","bill_amount",
    "payout_amount","item_status","bill_date","submission_date"]].head()

object            31
float64           20
datetime64[ns]    12
int64              6
Int64              5
bool               4
category           1
Name: count, dtype: int64


,item_id,employee_id,expense_category_code,bill_amount,payout_amount,item_status,bill_date,submission_date
0,e-31,794,e,170.0,0.0,Rejected,2026-07-16,2026-07-16 10:19:44
1,LEARN-10,794,LEARN,22.0,0.0,Accepted,2026-07-01,2026-07-16 10:12:22
2,LEARN-9,794,LEARN,50.0,0.0,Accepted,2026-07-01,2026-07-16 10:12:22
3,LEARN-8,794,LEARN,11.0,0.0,Accepted,2026-07-09,2026-07-16 10:10:51
4,LEARN-7,794,LEARN,11.0,0.0,Accepted,2026-07-01,2026-07-16 10:10:51


In [10]:
#feature creation
df["is_approved"] = df["item_status"].isin(["Accepted", "Paid"])
# can remove this
df["is_clean_approval"] = (
    df["is_approved"] & (df["auto_approved"] != True) & (df["override_count"].fillna(0) == 0)
)

df["submit_lag_days"] = (df["submission_date"] - df["bill_date"]).dt.days
df["bill_period"] = df["bill_date"].dt.to_period("M")

df["is_mileage"] = df["mileage_quantity"].notna()

print(df[["is_approved","is_clean_approval","is_mileage"]].sum())
print("submit_lag_days:")
print(df["submit_lag_days"].describe())

is_approved          63
is_clean_approval    35
is_mileage           25
dtype: int64
submit_lag_days:
count     83.000000
mean       9.879518
std       25.077634
min        0.000000
25%        0.000000
50%        0.000000
75%        7.500000
max      115.000000
Name: submit_lag_days, dtype: float64


In [14]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df, minimal=True, title="EC-1062 Raw Claims Profile")
profile.to_file("reports/raw_profile.html")

Summarize dataset:  17%|▏| 15/90 [00:00<00:01, 38.80it/s, Describe
Summarize dataset:  31%|▎| 28/90 [00:00<00:01, 51.58it/s, Describe
Summarize dataset:  52%|▌| 47/90 [00:00<00:00, 62.61it/s, Describe
Summarize dataset:  66%|▋| 59/90 [00:01<00:00, 36.82it/s, Describe
Summarize dataset:  71%|▋| 64/90 [00:01<00:00, 35.16it/s, Describe
Summarize dataset:  90%|▉| 81/90 [00:01<00:00, 57.95it/s, Describe
Summarize dataset:  92%|▉| 83/90 [00:01<00:00, 61.32it/s, Describe
100%|█████████████████████████████| 85/85 [00:01<00:00, 68.49it/s]
Summarize dataset: 100%|█| 91/91 [00:01<00:00, 54.94it/s, Complete
Export report to file: 100%|████████| 1/1 [00:00<00:00, 40.25it/s]


In [13]:
import os
os.makedirs("data/interim", exist_ok=True)
df.to_parquet("data/interim/claims_typed.parquet", index=False)
print("saved:", df.shape)

saved: (83, 84)


In [14]:
import sys; sys.path.append("..")
from src.profiles import build_profiles, is_eligible
from src import config

df = pd.read_parquet("../data/interim/claims_typed.parquet")
profiles = build_profiles(df)
profiles["eligible"] = profiles.apply(is_eligible, axis=1)

#print(f"eligible under MIN_PRIOR={config.MIN_PRIOR_APPROVALS}:",
#      profiles["eligible"].sum(), "of", len(profiles))
profiles.to_parquet("../data/interim/profiles.parquet", index=False)
profiles.sort_values("n_prior_approved").head(10)

eligible under MIN_PRIOR=3: 2 of 17


,employee_id,expense_category_code,n_prior_approved,centre,spread,cv,median_gap_days,months_claiming,prior_rejection,eligible
8,8191,DM,7,262.000,56.338800,0.215034,0.5,2.003942,True,True
9,8191,F,4,10.175,4.929645,0.484486,0.0,0.689882,True,False
1,1,LD,4,55.000,66.717000,1.213036,3.0,6.438896,False,True
3,17,F,3,100.000,140.847000,1.408470,0.0,0.000000,False,False
0,1,DM,2,600.000,593.040000,0.988400,42.0,1.379763,True,False
14,8192,F,2,310.000,429.954000,1.386948,100.0,3.285151,False,False
15,8193,IB,2,234.000,296.520000,1.267179,0.0,0.000000,False,False
2,17,DM,2,150.000,74.130000,0.494200,0.0,0.000000,False,False
4,794,ET,1,0.000,0.000000,0.000000,NaN,0.000000,False,False
7,794,LEARN,1,0.000,0.000000,0.000000,NaN,0.000000,False,False
